<a href="https://colab.research.google.com/github/codebysumit/cryptography-algorithms/blob/master/notebooks/columnar_transposition_multi_round.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Columnar Transposition Technique (Multiple Rounds)

## History
The Columnar Transposition Cipher is a classical transposition cipher. It does not change the letters of the message like a substitution cipher does. It only changes the position of the letters. Militaries used this technique a lot in the early 20th century, including in World War I and World War II, because it is easy to do by hand but still hard to break without the key.

## What is Columnar Transposition Technique?
In this cipher, the plaintext is written into a grid of rows and columns. The number of columns is equal to the length of the keyword. Each column gets a rank based on the alphabetical order of the letters in the keyword. The ciphertext is created by reading the columns in that rank order, not in the normal left to right order.

When this whole process is repeated many times, one after another, using a different keyword each time, it is called **Multiple Round Columnar Transposition**. Doing more than one round mixes the letter positions again and again, which makes the cipher much stronger than a single round.

## Cryptography Algorithm

### Constants
*   **key**: The keyword used to decide the column order for one round.
*   $n$ = length of **key** = number of columns in the grid.
*   $rows$ = number of rows in the grid = $\text{len(text)} / n$.
*   **pad_char**: Filler letter added at the end of the plaintext if it does not fill the grid exactly. We use `X`.
*   **keys**: A list of keywords, one keyword for each round.

### 1. Padding
Before the first round starts, the plaintext length must be a multiple of $n$. If it is not, `pad_char` is added at the end until it is.

### 2. Encryption (Single Round)
1.  Write the text into a grid, row by row, using $n$ columns.
2.  Work out the column order from the keyword. This order comes from sorting the letters of the keyword alphabetically.
3.  Read the grid column by column, following that order, and join the letters together. This is the ciphertext for one round.

### 3. Multiple Round Encryption
$$C_1 = E(P, K_1)$$
$$C_2 = E(C_1, K_2)$$
$$C_n = E(C_{n-1}, K_n)$$

Here $P$ is the plaintext, $K_1, K_2, \dots, K_n$ are the keywords for each round, and $C_n$ is the final ciphertext after all rounds are done.

### 4. Decryption (Single Round)
1.  Work out the column order from the keyword, the same way as encryption.
2.  Split the ciphertext into $n$ columns, each of length $rows$, based on that column order.
3.  Rebuild the grid by placing each column back into its correct position.
4.  Read the grid row by row to get back the text.

### 5. Multiple Round Decryption
To decrypt, the rounds must be undone in reverse order:
$$C_{n-1} = D(C_n, K_n)$$
$$C_{n-2} = D(C_{n-1}, K_{n-1})$$
$$P = D(C_1, K_1)$$

### Key Requirements
*   **Same length**: All keywords in every round must have the same length, so the grid size stays the same in every round.
*   **Unique ranking**: Repeated letters in a keyword are ranked by their order of appearance (1st, 2nd, ...), so the column order is always clear.
*   **More rounds, more strength**: Using more rounds and longer keywords makes the cipher harder to break.

### 1. Import Dependencies

In [1]:
import random
import string


### 2. Helper Utilities

In [2]:
def get_column_order(key: str):
    # rank each column of the key in alphabetical order
    # repeated letters are ranked by their first appearance
    indexed = sorted(list(enumerate(key)), key=lambda pair: (pair[1], pair[0]))
    order = [0] * len(key)
    for rank, (original_index, char) in enumerate(indexed):
        order[original_index] = rank
    return order


def pad_text(text: str, n: int, pad_char: str = 'X') -> str:
    # add filler letters so the text length becomes a multiple of n
    remainder = len(text) % n
    if remainder == 0:
        return text
    pad_len = n - remainder
    return text + (pad_char * pad_len)


### 3. Generate Random Keys

In [3]:
def generate_keys(num_rounds: int = 2, key_length: int = 6):
    # build one random keyword for each round, all of the same length
    keys = []
    for _ in range(num_rounds):
        letters = list(string.ascii_uppercase)
        random.shuffle(letters)
        key = ''.join(letters[:key_length])
        keys.append(key)
    return keys


### 4. Encryption (Single Round)

In [4]:
def encrypt_round(text: str, key: str) -> str:
    n = len(key)
    rows = len(text) // n

    # step 1: write the text into a grid, n columns wide
    grid = [text[i * n:(i + 1) * n] for i in range(rows)]

    # step 2: work out the column order from the key
    order = get_column_order(key)

    # step 3: read the grid column by column, in rank order
    cipher = [''] * n
    for col in range(n):
        rank = order[col]
        cipher[rank] = ''.join(grid[row][col] for row in range(rows))

    return ''.join(cipher)


### 5. Multiple Round Encryption

In [5]:
def encrypt_multi(text: str, keys, pad_char: str = 'X') -> str:
    # pad once, using the length of the first key
    n = len(keys[0])
    result = pad_text(text, n, pad_char)

    # run one full round of encryption for every key in the list
    for key in keys:
        result = encrypt_round(result, key)

    return result


### 6. Decryption (Single Round)

In [6]:
def decrypt_round(cipher: str, key: str) -> str:
    n = len(key)
    rows = len(cipher) // n
    order = get_column_order(key)

    # step 1 and 2: cut the ciphertext back into its n columns
    columns = [''] * n
    pos = 0
    for rank in range(n):
        original_col = order.index(rank)
        columns[original_col] = cipher[pos:pos + rows]
        pos += rows

    # step 3 and 4: rebuild the grid and read it row by row
    grid = []
    for row in range(rows):
        grid.append(''.join(columns[col][row] for col in range(n)))

    return ''.join(grid)


### 7. Multiple Round Decryption

In [7]:
def decrypt_multi(cipher: str, keys) -> str:
    result = cipher

    # undo the rounds in reverse order
    for key in reversed(keys):
        result = decrypt_round(result, key)

    return result


### 8. Example usage

In [16]:
# random.seed(1)
keys = generate_keys(num_rounds=3, key_length=6)
print("Generated keys:", keys)

Generated keys: ['MLZCIY', 'LUCNWH', 'OAWSDH']


In [18]:
plaintext = """TOP secret Massage! Agent 101, visit Area 51 (37d14'0\"N 115d48'30\"W)."""
print(f"Original Plain Text: {plaintext}")

cipher_text = encrypt_multi(plaintext, keys)
print("Encrypted:", cipher_text)

decrypted_text = decrypt_multi(cipher_text, keys)
print("Decrypted (raw):", decrypted_text)

decrypted_final = decrypted_text.rstrip('X')
print("Decrypted (final):", decrypted_final)

match = plaintext == decrypted_final
print(f"Verification Match: {match}")

Original Plain Text: TOP secret Massage! Agent 101, visit Area 51 (37d14'0"N 115d48'30"W).
Encrypted: 'dXWe 1N(534ssviOP Are 114).Ar 11 8'aas  Tg!tc0t0"XXa 5d370"geitseen M1,
Decrypted (raw): TOP secret Massage! Agent 101, visit Area 51 (37d14'0"N 115d48'30"W).XXX
Decrypted (final): TOP secret Massage! Agent 101, visit Area 51 (37d14'0"N 115d48'30"W).
Verification Match: True
